In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import SolverFactory
from pyomo.environ import *


In [ ]:

compostos = ['um','dois','tres']

conter= ['enxofre','ferro','potassio']

minimos = {
    conter[0]:0.20,
    conter[1]:0.30,
    conter[2]:0.30,
}


# potassio nao pode ultrapassar 0.45

#a soma dos Conter (compostos) maior igual 600

informacoes = {
    compostos[0]:{
        conter[0]: 0.20,
        conter[1]: 0.60,
        conter[2]:0.20,
    },
    compostos[1]:{
        conter[0]: 0.40,
        conter[1]: 0.30,
        conter[2]:0.30,
    },
    compostos[2]:{
        conter[0]: 0.10,
        conter[1]: 0.40,
        conter[2]:0.50,
    },
}

custos = {
    compostos[0]:5,
    compostos[1]: 5.25,
    compostos[2]: 5.50
}

In [38]:
model = pyo.ConcreteModel()

model.compostos = pyo.Set(initialize = compostos)
model.itens = pyo.Set(initialize = conter)

model.x = pyo.Var(model.compostos, bounds=(0, None), domain=pyo.NonNegativeIntegers)

def obj(model):
    return sum(
        model.x[c]* custos[c]  for c in model.compostos
    )
model.objetivo = pyo.Objective(rule=obj, sense=pyo.minimize)

In [39]:
def restricao_qnt(model, i):
    return sum(
        model.x[c]*informacoes[c][i]   for c in model.compostos
    ) >= minimos[i] * sum(model.x[c] for c in model.compostos)
model.res_qnt = pyo.Constraint(model.itens , rule=restricao_qnt)

def potassio(model):
    return sum(model.x[c]* informacoes[c]['potassio'] for c in model.compostos) <= 0.45 * sum(model.x[c] for c in model.compostos)
model.res_potassio  =pyo.Constraint(rule=potassio)

def soma_total(model):
    return sum(model.x[c] for c in model.compostos) >= 600
model.soma = pyo.Constraint(rule=soma_total)

In [40]:
model.pprint()

2 Set Declarations
    compostos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'um', 'dois', 'tres'}
    itens : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'enxofre', 'ferro', 'potassio'}

1 Var Declarations
    x : Size=3, Index=compostos
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        dois :     0 :  None :  None : False :  True : NonNegativeIntegers
        tres :     0 :  None :  None : False :  True : NonNegativeIntegers
          um :     0 :  None :  None : False :  True : NonNegativeIntegers

1 Objective Declarations
    objetivo : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 5*x[um] + 5.25*x[dois] + 5.5*x[tres]

3 Constraint Declarations
    res_potassio : Size=1, Index=None, Active=True
        Key  : Lower : Body              

In [41]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmpte1knuo0.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpkpaedsoy.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpkpaedsoy.pyomo.lp
Objective sense      : Minimize
Variables            :       3  [General Integer: 3]
Objective nonzeros   :       3
Linear constraints   :       5  [Less: 4,  Greater: 1]
  Nonzeros           :      12
  RHS nonzeros       :       1

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 5.

In [43]:
for m in model.x:
    print(f"Quantidade de Composto {m}: {pyo.value(model.x[m])}")
print(f"Custo: {pyo.value(model.objetivo)}")

Quantidade de Composto um: 342.0
Quantidade de Composto dois: 87.0
Quantidade de Composto tres: 171.0
Custo: 3107.25
